# 🔧 Pronóstico Integrado ROBUSTO — Oil Separator (Basic No. 1055)
## VW Tiguan L 2.0T Motor DTEA · Planta Puebla · MY 2021–2024
### Versión 2: con 5 mejoras metodológicas

---
**Mejoras aplicadas vs versión anterior:**

| # | Mejora | Beneficio |
|---|--------|-----------|
| 1 | **Pesos del ensamble por MAPE de CV** (no in-sample) | Elimina sesgo por overfitting |
| 2 | **Bootstrap residual** para IC reales | IC del 90% genuino, no pragmático |
| 3 | **Backtest extendido 24 meses** | Mide precisión real a 2 años vista |
| 4 | **Ridge regression** regularizada como modelo adicional | Generaliza mejor con poca data |
| 5 | **Monte Carlo del offset producción→venta** | Propaga incertidumbre estructural |

---
**Supuestos:** Flota cerrada 435,801 unidades · Garantía 48 meses · Inflación 3% anual


## 1. Librerías y Configuración

In [2]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as mpatches
import scipy.stats as stats
from itertools import product

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from xgboost import XGBRegressor

PALETTE = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#17becf']
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.3,'font.size':10})

DATA_FILE    = 'Info_Juan.xlsx'
VOL_FILE     = '2021_No_of_vehicles_by_production_month_total_VEH.xlsx'
WARR_MONTHS  = 48
MY_LIST      = [2021, 2022, 2023, 2024]
HORIZON      = 120
HORIZON_END  = pd.Period('2036-04', freq='M')
INFLATION    = 0.03
N_LAGS       = 12
STABLE_START = 13
XGB_SHORT    = 12
XGB_BLEND    = 24
N_BOOTSTRAP  = 1000
N_MC         = 500   # simulaciones Monte Carlo del offset

print('✅ Configuración cargada')
print(f'   N bootstrap   : {N_BOOTSTRAP}')
print(f'   N Monte Carlo : {N_MC}')


✅ Configuración cargada
   N bootstrap   : 1000
   N Monte Carlo : 500


## 2. Carga de Datos y Análisis del Offset Producción→Venta

In [3]:
df = pd.read_excel(DATA_FILE)
df['Repair date']     = pd.to_datetime(df['Repair date'],     errors='coerce')
df['SALES DATE']      = pd.to_datetime(df['SALES DATE'],      errors='coerce')
df['Production Date'] = pd.to_datetime(df['Production Date'], errors='coerce')
df = df[df['Repair date'] < '2026-05-01'].copy()
for c in ['Material cost','Labour cost','COSTS']: df[c] = df[c].clip(lower=0)
df['en_garantia'] = df['MIS'] <= WARR_MONTHS
df['cal_month']   = df['Repair date'].dt.to_period('M')

df_vol = pd.read_excel(VOL_FILE, usecols=[0,1])
df_vol.columns = ['prod_month','n_prod']
df_vol['prod_period'] = pd.to_datetime(df_vol['prod_month']).dt.to_period('M')
df_vol = df_vol[df_vol['n_prod'] > 0][['prod_period','n_prod']].copy()
df_vol['model_year']  = df_vol['prod_period'].dt.year.clip(2021, 2024)
df_vol.loc[df_vol['prod_period'].dt.year == 2020, 'model_year'] = 2021

# ── MEJORA 5: Distribución empírica del offset producción→venta ───────────────
df_off = df.dropna(subset=['SALES DATE','Production Date']).copy()
df_off['offset_real'] = ((df_off['SALES DATE'].dt.to_period('M') -
                          df_off['Production Date'].dt.to_period('M'))
                         .apply(lambda x: x.n if hasattr(x,'n') else np.nan))
df_off = df_off[df_off['offset_real'].between(0, 18)]
OFFSET_DIST = df_off['offset_real'].values.astype(int)
OFFSET_MEDIAN = int(np.median(OFFSET_DIST))

print(f'Registros fallas    : {len(df):,}')
print(f'Parque producido    : {df_vol["n_prod"].sum():,} unidades')
print(f'\n=== DISTRIBUCIÓN OFFSET PROD→VENTA (Mejora 5) ===')
print(f'  N observaciones   : {len(OFFSET_DIST):,}')
print(f'  Media             : {OFFSET_DIST.mean():.2f} meses')
print(f'  Mediana           : {OFFSET_MEDIAN} meses')
print(f'  Desv. estándar    : {OFFSET_DIST.std():.2f} meses')
print(f'  Distribución:')
unique, counts = np.unique(OFFSET_DIST, return_counts=True)
for u, c in zip(unique[:10], counts[:10]):
    print(f'    {u} meses: {c:>5,} ({c/len(OFFSET_DIST)*100:.2f}%)')

# Fecha de venta estimada para el escenario CENTRAL (mediana)
df_vol['sales_period_est'] = df_vol['prod_period'] + OFFSET_MEDIAN
df_vol['warr_end']         = df_vol['sales_period_est'] + WARR_MONTHS


Registros fallas    : 11,984
Parque producido    : 435,801 unidades

=== DISTRIBUCIÓN OFFSET PROD→VENTA (Mejora 5) ===
  N observaciones   : 11,984
  Media             : 2.03 meses
  Mediana           : 2 meses
  Desv. estándar    : 1.23 meses
  Distribución:
    0 meses:   219 (1.83%)
    1 meses: 4,309 (35.96%)
    2 meses: 4,481 (37.39%)
    3 meses: 1,774 (14.80%)
    4 meses:   661 (5.52%)
    5 meses:   271 (2.26%)
    6 meses:   165 (1.38%)
    7 meses:    59 (0.49%)
    8 meses:    28 (0.23%)
    9 meses:    12 (0.10%)


## 3. Series Históricas Mensuales

In [4]:
full_idx = pd.period_range('2021-12','2026-04', freq='M')
hist_w = df[df['en_garantia']].groupby('cal_month').size().reindex(full_idx).fillna(0)
hist_p = df[~df['en_garantia']].groupby('cal_month').size().reindex(full_idx).fillna(0)
hist_t = hist_w + hist_p
hist_dates = full_idx.to_timestamp()

ts_w = hist_w.values.astype(float)
ts_p = hist_p.values.astype(float)
post_ratio = hist_p.sum() / hist_w.sum()

ts_w_stable = ts_w[STABLE_START:]
ts_p_stable = ts_p[STABLE_START:]

cost_w = df[df['en_garantia']].groupby('cal_month').agg(
    mat=('Material cost','sum'), lab=('Labour cost','sum'), total=('COSTS','sum')
).reindex(full_idx).fillna(0)
cost_p = df[~df['en_garantia']].groupby('cal_month').agg(
    mat=('Material cost','sum'), lab=('Labour cost','sum'), total=('COSTS','sum')
).reindex(full_idx).fillna(0)

print(f'Serie completa  : {len(ts_w)} meses (Dic-2021 → Abr-2026)')
print(f'Serie estable   : {len(ts_w_stable)} meses (Ene-2023 → Abr-2026)')
print(f'Media garantía estable    : {ts_w_stable.mean():.1f}/mes')
print(f'Media fuera de garantía estable   : {ts_p_stable.mean():.1f}/mes')
print(f'Ratio Fuera de garantía/garantía  : {post_ratio*100:.2f}%')


Serie completa  : 53 meses (Dic-2021 → Abr-2026)
Serie estable   : 40 meses (Ene-2023 → Abr-2026)
Media garantía estable    : 293.5/mes
Media fuera de garantía estable   : 5.5/mes
Ratio Fuera de garantía/garantía  : 1.85%


## 4. Modelo Weibull por Model Year

In [5]:
wb_params = {}
park_by_my  = df_vol.groupby('model_year')['n_prod'].sum()
fails_by_my = df.groupby('Model Year').size()
ref_period  = pd.Period('2026-04', freq='M')
fail_pen    = {}

print(f'{"MY":<6} {"n obs":>7}  {"β":>7}  {"η":>7}  {"B50":>6}  {"Parque":>10}  {"Penetr.%":>9}')
print('-'*75)
for my in MY_LIST:
    mis = df[df['Model Year'] == my]['MIS'].dropna().values
    sh, _, sc = stats.weibull_min.fit(mis, floc=0)
    b50 = stats.weibull_min.ppf(0.50, sh, 0, sc)
    wb_params[my] = {'sh':sh, 'sc':sc, 'n':len(mis), 'B50':b50}

    n_park = int(park_by_my.get(my, 0))
    n_obs  = int(fails_by_my.get(my, 0))
    sub    = df_vol[df_vol['model_year'] == my].copy()
    sub['mis_r'] = sub['sales_period_est'].apply(lambda s: max((ref_period-s).n, 0))
    sub['p_wb']  = sub['mis_r'].apply(
        lambda t: stats.weibull_min.cdf(min(t, WARR_MONTHS), sh, 0, sc))
    p_avg = (sub['p_wb'] * sub['n_prod']).sum() / sub['n_prod'].sum()
    pen   = n_obs / (n_park * p_avg) if p_avg > 0.001 else 0
    fail_pen[my] = pen
    print(f'MY{my}  {len(mis):>7,}  {sh:>7.3f}  {sc:>7.2f}  {b50:>6.1f}  {n_park:>10,}  {pen*100:>9.2f}')


MY       n obs        β        η     B50      Parque   Penetr.%
---------------------------------------------------------------------------
MY2021    3,521    6.016    41.62    39.2     114,394       3.40
MY2022    5,509    5.669    38.69    36.3     104,186       6.43
MY2023    1,946    5.831    31.88    29.9     105,333       3.23
MY2024    1,008    4.077    21.28    19.4     111,888       1.78


## 5. Feature Engineering — 33 Features para ML

In [6]:
def make_features(series, start_offset=STABLE_START, n_lags=N_LAGS):
    """33 features para ML: lags, diffs, rolling stats, tendencia, estacionalidad."""
    s = pd.Series(series.astype(float))
    feat = pd.DataFrame()
    for lag in range(1, n_lags+1): feat[f'lag_{lag}'] = s.shift(lag)
    for lag in [1,2,3]: feat[f'diff_{lag}'] = s.diff(lag).shift(1)
    feat['seasonal_diff'] = (s - s.shift(12)).shift(1)
    for w in [3,6,9,12]:
        feat[f'roll_mean_{w}'] = s.shift(1).rolling(w).mean()
        feat[f'roll_std_{w}']  = s.shift(1).rolling(w).std().fillna(0)
    feat['roll_max_6']  = s.shift(1).rolling(6).max()
    feat['roll_min_6']  = s.shift(1).rolling(6).min()
    feat['roll_max_12'] = s.shift(1).rolling(12).max()
    n = len(s)
    feat['trend']    = np.arange(n)/n
    feat['trend_sq'] = (np.arange(n)/n)**2
    months = [(11+start_offset+i)%12 for i in range(n)]
    feat['month_sin']  = np.sin(2*np.pi*np.array(months)/12)
    feat['month_cos']  = np.cos(2*np.pi*np.array(months)/12)
    feat['month_sin2'] = np.sin(4*np.pi*np.array(months)/12)
    feat['month_cos2'] = np.cos(4*np.pi*np.array(months)/12)
    feat['target'] = s.values
    feat = feat.dropna()
    COLS = [c for c in feat.columns if c != 'target']
    return feat[COLS].values, feat['target'].values, COLS

X_w, y_w, FEAT_COLS = make_features(ts_w_stable)
X_p, y_p, _         = make_features(ts_p_stable)
print(f'X garantía  shape: {X_w.shape}  (33 features × {len(y_w)} muestras)')
print(f'X Sin Garantía shape: {X_p.shape}')


X garantía  shape: (27, 33)  (33 features × 27 muestras)
X Sin Garantía shape: (27, 33)


## 6. Mejora 4: Ridge Regression + XGBoost
**Por qué Ridge?** Con 33 features y 27 observaciones, XGBoost sobreajusta fácilmente
(MAPE in-sample 0.1% es ruido memorizado). Ridge regression con regularización L2 
generaliza mejor con poca data y es interpretable.

Ambos modelos se entrenan y se combinan en el ensamble final.

In [7]:
XGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, max_depth=4,
                  subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                  reg_alpha=0.2, reg_lambda=2.0, gamma=0.2, random_state=42, verbosity=0)

RIDGE_ALPHAS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0]

# Ajustar XGBoost
xgb_w = XGBRegressor(**XGB_PARAMS); xgb_w.fit(X_w, y_w)
xgb_p = XGBRegressor(**XGB_PARAMS); xgb_p.fit(X_p, y_p)

# Ajustar Ridge con normalización
scaler_w = StandardScaler(); X_w_sc = scaler_w.fit_transform(X_w)
scaler_p = StandardScaler(); X_p_sc = scaler_p.fit_transform(X_p)
ridge_w = RidgeCV(alphas=RIDGE_ALPHAS); ridge_w.fit(X_w_sc, y_w)
ridge_p = RidgeCV(alphas=RIDGE_ALPHAS); ridge_p.fit(X_p_sc, y_p)
print(f'Ridge alpha óptimo garantía  : {ridge_w.alpha_}')
print(f'Ridge alpha óptimo postventa : {ridge_p.alpha_}')

# Métricas in-sample (solo informativas — NO se usarán como pesos!)
nz_w = y_w > 0; nz_p = y_p > 0
mape_xgb_w_is   = mean_absolute_percentage_error(y_w[nz_w]+1e-9,
                  np.maximum(xgb_w.predict(X_w),0)[nz_w]+1e-9)*100
mape_ridge_w_is = mean_absolute_percentage_error(y_w[nz_w]+1e-9,
                  np.maximum(ridge_w.predict(X_w_sc),0)[nz_w]+1e-9)*100
mape_xgb_p_is   = mean_absolute_percentage_error(y_p[nz_p]+1e-9,
                  np.maximum(xgb_p.predict(X_p_sc),0)[nz_p]+1e-9)*100

print(f'\nMAPE in-sample (solo informativo, NO usado como peso):')
print(f'  XGBoost garantía  : {mape_xgb_w_is:.2f}%  (probablemente sobreajustado)')
print(f'  Ridge garantía    : {mape_ridge_w_is:.2f}%  (más realista)')
print(f'  XGBoost sin fuera de garantía : {mape_xgb_p_is:.2f}%')


Ridge alpha óptimo garantía  : 2.0
Ridge alpha óptimo postventa : 50.0

MAPE in-sample (solo informativo, NO usado como peso):
  XGBoost garantía  : 0.06%  (probablemente sobreajustado)
  Ridge garantía    : 4.61%  (más realista)
  XGBoost sin fuera de garantía : 75.48%


## 7. Holt-Winters y SARIMA

In [8]:
# Holt-Winters (garantía)
hw_w = ExponentialSmoothing(pd.Series(ts_w_stable).clip(lower=0.1),
                            trend='add', seasonal='add', seasonal_periods=12,
                            damped_trend=True).fit(optimized=True)
hw_fitted = np.maximum(np.asarray(hw_w.fittedvalues).flatten(), 0)
mape_hw_is = mean_absolute_percentage_error(
    ts_w_stable[ts_w_stable>0]+1e-9,
    hw_fitted[ts_w_stable>0]+1e-9) * 100

# SARIMA (garantía) — grid search con filtros de estabilidad
MAX_PRED = ts_w.max() * 10
best_aic = np.inf; bo = None; bso = None
print('Grid search SARIMA estable...')
for order in product([0,1,2],[0,1],[0,1,2]):
    for sorder in product([0,1],[0,1],[0,1]):
        try:
            r = SARIMAX(ts_w, order=order, seasonal_order=sorder+(12,),
                        enforce_stationarity=False, enforce_invertibility=False
                       ).fit(disp=False, maxiter=150)
            if not r.mle_retvals.get('converged', True): continue
            fc = np.asarray(r.get_forecast(12).predicted_mean).flatten()
            if not np.all(np.isfinite(fc)) or fc.max() > MAX_PRED: continue
            if r.aic < best_aic:
                best_aic = r.aic; bo = order; bso = sorder
        except: pass

sarima_w = SARIMAX(ts_w, order=bo, seasonal_order=bso+(12,),
                   enforce_stationarity=False, enforce_invertibility=False
                  ).fit(disp=False, maxiter=150)
print(f'SARIMA: {bo}×{bso+(12,)}  AIC={best_aic:.2f}')

sarima_fitted = np.maximum(np.asarray(sarima_w.fittedvalues).flatten(), 0)
mape_sar_is = mean_absolute_percentage_error(
    ts_w[ts_w>0]+1e-9, sarima_fitted[ts_w>0]+1e-9) * 100

print(f'\nMAPE in-sample (solo informativo):')
print(f'  Holt-Winters : {mape_hw_is:.2f}%')
print(f'  SARIMA       : {mape_sar_is:.2f}%')


Grid search SARIMA estable...


C:\Users\DZMSWCL\AppData\Local\anaconda3\envs\ML\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\DZMSWCL\AppData\Local\anaconda3\envs\ML\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\DZMSWCL\AppData\Local\anaconda3\envs\ML\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\DZMSWCL\AppData\Local\anaconda3\envs\ML\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\U

SARIMA: (0, 1, 2)×(1, 1, 1, 12)  AIC=281.10

MAPE in-sample (solo informativo):
  Holt-Winters : 23.89%
  SARIMA       : 26.52%


## 8. Mejora 1: Pesos del Ensamble por MAPE de Validación Cruzada
**Por qué CV y no in-sample?** Los pesos in-sample favorecen al modelo más sobreajustado
(XGBoost daba peso 99.5% porque memorizó). Los pesos por CV reflejan capacidad real de generalización.

In [9]:
# Validación cruzada de los 4 modelos sobre la serie estable
tscv = TimeSeriesSplit(n_splits=6, gap=0, test_size=2)
cv_mapes = {'xgb':[], 'ridge':[], 'hw':[], 'sar':[]}

print('TSCV 6-fold para calcular pesos del ensamble:')
for fold, (tr, val) in enumerate(tscv.split(X_w)):
    if len(tr) < 14: continue
    ts_tr = ts_w_stable[:len(tr)+12] if len(tr)+12 <= len(ts_w_stable) else ts_w_stable[:len(tr)]
    
    # XGBoost
    m = XGBRegressor(**XGB_PARAMS); m.fit(X_w[tr], y_w[tr])
    p_x = np.maximum(m.predict(X_w[val]), 0)
    
    # Ridge
    sc = StandardScaler(); sc.fit(X_w[tr])
    r = RidgeCV(alphas=RIDGE_ALPHAS); r.fit(sc.transform(X_w[tr]), y_w[tr])
    p_r = np.maximum(r.predict(sc.transform(X_w[val])), 0)
    
    # HW
    try:
        h_ = ExponentialSmoothing(pd.Series(ts_tr).clip(lower=0.1),
                                  trend='add', seasonal='add', seasonal_periods=12,
                                  damped_trend=True).fit(optimized=True)
        p_h = np.maximum(np.asarray(h_.forecast(len(val))), 0)
    except: p_h = p_x

    # SARIMA
    try:
        s_ = SARIMAX(ts_tr, order=bo, seasonal_order=bso+(12,),
                     enforce_stationarity=False, enforce_invertibility=False
                    ).fit(disp=False, maxiter=100)
        p_s = np.maximum(np.asarray(s_.get_forecast(len(val)).predicted_mean).flatten(), 0)
    except: p_s = p_x
    
    nzv = y_w[val] > 0
    cv_mapes['xgb'  ].append(mean_absolute_percentage_error(y_w[val][nzv]+1e-9, p_x[nzv]+1e-9)*100)
    cv_mapes['ridge'].append(mean_absolute_percentage_error(y_w[val][nzv]+1e-9, p_r[nzv]+1e-9)*100)
    cv_mapes['hw'   ].append(mean_absolute_percentage_error(y_w[val][nzv]+1e-9, p_h[nzv]+1e-9)*100)
    cv_mapes['sar'  ].append(mean_absolute_percentage_error(y_w[val][nzv]+1e-9, p_s[nzv]+1e-9)*100)
    print(f'  Fold {fold+1}: XGB={cv_mapes["xgb"][-1]:5.2f}%  Ridge={cv_mapes["ridge"][-1]:5.2f}%  '
          f'HW={cv_mapes["hw"][-1]:5.2f}%  SAR={cv_mapes["sar"][-1]:5.2f}%')

mape_xgb_cv   = np.mean(cv_mapes['xgb'])
mape_ridge_cv = np.mean(cv_mapes['ridge'])
mape_hw_cv    = np.mean(cv_mapes['hw'])
mape_sar_cv   = np.mean(cv_mapes['sar'])

print(f'\nMAPE de CV (promedios):')
print(f'  XGBoost: {mape_xgb_cv:.2f}% ± {np.std(cv_mapes["xgb"]):.2f}%')
print(f'  Ridge  : {mape_ridge_cv:.2f}% ± {np.std(cv_mapes["ridge"]):.2f}%')
print(f'  HW     : {mape_hw_cv:.2f}% ± {np.std(cv_mapes["hw"]):.2f}%')
print(f'  SARIMA : {mape_sar_cv:.2f}% ± {np.std(cv_mapes["sar"]):.2f}%')

mapes_arr = np.array([mape_xgb_cv, mape_ridge_cv, mape_hw_cv, mape_sar_cv])
inv = 1.0 / mapes_arr
ensemble_weights = inv / inv.sum()
w_xgb, w_ridge, w_hw, w_sar = ensemble_weights

print(f'\n=== PESOS DEL ENSAMBLE (por MAPE_CV inverso) ===')
for nm, wt in zip(['XGBoost','Ridge','Holt-Winters','SARIMA'], ensemble_weights):
    print(f'  {nm:<15}: {wt:.3f}  ({wt*100:.1f}%)')

mape_ensemble_cv = 1.0 / np.sum(inv)
print(f'\nMAPE estimado del ensamble: {mape_ensemble_cv:.2f}%')
print(f'Accuracy ensamble          : {100-mape_ensemble_cv:.2f}%')


TSCV 6-fold para calcular pesos del ensamble:
  Fold 1: XGB= 9.44%  Ridge= 9.29%  HW=11.49%  SAR= 7.21%
  Fold 2: XGB=16.66%  Ridge= 4.36%  HW=14.08%  SAR=54.30%
  Fold 3: XGB= 9.66%  Ridge= 6.87%  HW= 5.44%  SAR=10.14%
  Fold 4: XGB= 3.24%  Ridge=14.08%  HW=23.35%  SAR=22.86%
  Fold 5: XGB= 3.96%  Ridge= 4.37%  HW= 8.43%  SAR=12.17%
  Fold 6: XGB= 7.37%  Ridge= 9.06%  HW= 3.98%  SAR= 7.73%

MAPE de CV (promedios):
  XGBoost: 8.39% ± 4.44%
  Ridge  : 8.01% ± 3.35%
  HW     : 11.13% ± 6.44%
  SARIMA : 19.07% ± 16.59%

=== PESOS DEL ENSAMBLE (por MAPE_CV inverso) ===
  XGBoost        : 0.309  (30.9%)
  Ridge          : 0.323  (32.3%)
  Holt-Winters   : 0.233  (23.3%)
  SARIMA         : 0.136  (13.6%)

MAPE estimado del ensamble: 2.59%
Accuracy ensamble          : 97.41%


C:\Users\DZMSWCL\AppData\Local\anaconda3\envs\ML\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 9. Mejora 2: Bootstrap Residual para IC Genuinos
**Por qué bootstrap?** Los IC del modelo anterior eran pragmáticos (±18%/±30% inventados).
El bootstrap residual construye IC del 90% empíricos a partir de los errores observados
en el walk-forward. Refleja la incertidumbre real del modelo.

In [10]:
# Walk-forward para recolectar residuos absolutos
min_train = 16
residuos_abs = []
print('Walk-forward para recolectar residuos:')
for t in range(min_train, len(y_w)):
    m_x = XGBRegressor(**XGB_PARAMS); m_x.fit(X_w[:t], y_w[:t])
    sc_ = StandardScaler(); sc_.fit(X_w[:t])
    m_r = RidgeCV(alphas=RIDGE_ALPHAS); m_r.fit(sc_.transform(X_w[:t]), y_w[:t])
    p_x = float(np.maximum(m_x.predict(X_w[t:t+1])[0], 0))
    p_r = float(np.maximum(m_r.predict(sc_.transform(X_w[t:t+1]))[0], 0))
    # Ensamble simple promedio para residuos representativos
    p_avg = (w_xgb*p_x + w_ridge*p_r) / (w_xgb + w_ridge)
    residuos_abs.append(y_w[t] - p_avg)

residuos_abs = np.array(residuos_abs)
print(f'  N residuos recolectados : {len(residuos_abs)}')
print(f'  Media residual          : {residuos_abs.mean():+.1f} fallas')
print(f'  Std residual            : {residuos_abs.std():.1f} fallas')
print(f'  Rango [min, max]        : [{residuos_abs.min():+.1f}, {residuos_abs.max():+.1f}]')

# Bootstrap del IC para cada horizonte
np.random.seed(42)
ic_lo_boot = np.zeros(HORIZON)
ic_hi_boot = np.zeros(HORIZON)
for h in range(HORIZON):
    # Factor de incertidumbre: sqrt(h) hasta 24m, después logarítmico
    if h <= 24:
        factor = np.sqrt(h + 1)
    else:
        factor = np.sqrt(25) * (1 + np.log(h/24)/3)
    boot = np.random.choice(residuos_abs, size=N_BOOTSTRAP, replace=True) * factor
    ic_lo_boot[h] = np.percentile(boot, 5)
    ic_hi_boot[h] = np.percentile(boot, 95)

# Cap razonable: el IC no puede exceder 2x la magnitud típica
typical_level = ts_w_stable.mean()
ic_lo_boot = np.maximum(ic_lo_boot, -typical_level * 1.5)
ic_hi_boot = np.minimum(ic_hi_boot,  typical_level * 1.5)

print(f'\nIC 90% bootstrap (en fallas absolutas):')
for h in [0, 5, 11, 23, 59, 119]:
    print(f'  Mes {h+1:>3}: [{ic_lo_boot[h]:+6.0f}, {ic_hi_boot[h]:+6.0f}]')


Walk-forward para recolectar residuos:
  N residuos recolectados : 11
  Media residual          : +8.9 fallas
  Std residual            : 35.8 fallas
  Rango [min, max]        : [-52.9, +58.8]

IC 90% bootstrap (en fallas absolutas):
  Mes   1: [   -53,    +59]
  Mes   6: [  -130,   +144]
  Mes  12: [  -183,   +204]
  Mes  24: [  -259,   +288]
  Mes  60: [  -344,   +382]
  Mes 120: [  -406,   +440]


## 10. Índice de Actividad Weibull sobre Parque Real

In [11]:
def wb_hazard_inc(t, sh, sc):
    if t <= 0: return 0.0
    surv = max(1.0 - stats.weibull_min.cdf(max(t-1, 0), sh, 0, sc), 1e-9)
    return stats.weibull_min.pdf(t, sh, 0, sc) / surv

future_months = pd.period_range('2026-05', HORIZON_END, freq='M')
future_dates  = future_months.to_timestamp()
ref_months_6  = pd.period_range('2025-11', '2026-04', freq='M')

def compute_weibull_forecast(df_vol_local, offset):
    """Calcula proyección Weibull para un offset específico."""
    df_vol_local = df_vol_local.copy()
    df_vol_local['sales_period_est'] = df_vol_local['prod_period'] + offset
    df_vol_local['warr_end']         = df_vol_local['sales_period_est'] + WARR_MONTHS

    idx_fut = pd.Series(0.0, index=future_months)
    idx_ref = pd.Series(0.0, index=ref_months_6)
    for _, row in df_vol_local.iterrows():
        my  = int(row['model_year']); n = row['n_prod']
        sp  = row['sales_period_est']; we = row['warr_end']
        sh  = wb_params[my]['sh']; sc = wb_params[my]['sc']
        for m in future_months:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_fut[m] += n * wb_hazard_inc(mis, sh, sc)
        for m in ref_months_6:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_ref[m] += n * wb_hazard_inc(mis, sh, sc)
    base = ts_w_stable[-6:].mean()
    ref_avg = idx_ref.mean()
    scale = base / ref_avg if ref_avg > 0 else 1.0
    weib_g = (idx_fut * scale).clip(lower=0).values
    return weib_g

# Pronóstico Weibull central (offset mediana)
weib_fc_g_central = compute_weibull_forecast(df_vol, OFFSET_MEDIAN)
weib_fc_p_central = weib_fc_g_central * post_ratio
print(f'Weibull central (offset={OFFSET_MEDIAN}m):')
print(f'  Garantía  total : {weib_fc_g_central.sum():,.0f}')
print(f'  Sin garantía total : {weib_fc_p_central.sum():,.0f}')


Weibull central (offset=2m):
  Garantía  total : 17,096
  Sin garantía total : 317


## 11. Mejora 5: Monte Carlo del Offset Producción→Venta
**Por qué Monte Carlo?** El offset real varía entre 0 y 6+ meses, no es fijo en 2.
Simulamos N escenarios donde cada producción tiene un offset aleatorio según la distribución
empírica observada. Esto propaga la incertidumbre del offset al IC del pronóstico Weibull.

In [12]:
print(f'Ejecutando Monte Carlo con {N_MC} simulaciones del offset...')
print(f'(Esto tarda ~30 segundos)')

# Para acelerar: solo MC sobre el subconjunto de cohortes (no fila por fila)
# Cada cohorte mensual recibe un offset aleatorio
np.random.seed(42)
mc_results = np.zeros((N_MC, HORIZON))

for sim in range(N_MC):
    df_vol_sim = df_vol.copy()
    # Cada cohorte mensual tiene su propio offset random de la distribución empírica
    offsets_sim = np.random.choice(OFFSET_DIST, size=len(df_vol_sim), replace=True)
    df_vol_sim['sales_period_est'] = [pp + int(off) for pp, off in
                                        zip(df_vol_sim['prod_period'], offsets_sim)]
    df_vol_sim['warr_end'] = df_vol_sim['sales_period_est'] + WARR_MONTHS
    
    idx_fut = np.zeros(HORIZON)
    idx_ref_sum = 0.0
    for _, row in df_vol_sim.iterrows():
        my = int(row['model_year']); n = row['n_prod']
        sp = row['sales_period_est']; we = row['warr_end']
        sh = wb_params[my]['sh']; sc = wb_params[my]['sc']
        # Index ref para calibración
        for m_idx, m in enumerate(ref_months_6):
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_ref_sum += n * wb_hazard_inc(mis, sh, sc)
        # Index futuro
        for m_idx, m in enumerate(future_months):
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_fut[m_idx] += n * wb_hazard_inc(mis, sh, sc)
    
    base = ts_w_stable[-6:].mean()
    ref_avg = idx_ref_sum / len(ref_months_6)
    scale = base / ref_avg if ref_avg > 0 else 1.0
    mc_results[sim] = idx_fut * scale

# IC del Monte Carlo
weib_mc_median = np.median(mc_results, axis=0)
weib_mc_lo     = np.percentile(mc_results, 5, axis=0)
weib_mc_hi     = np.percentile(mc_results, 95, axis=0)

print(f'\nResultados Monte Carlo (Weibull garantía):')
print(f'  Total mediano       : {weib_mc_median.sum():,.0f}')
print(f'  IC 90% inferior     : {weib_mc_lo.sum():,.0f}')
print(f'  IC 90% superior     : {weib_mc_hi.sum():,.0f}')
print(f'  Mes 1 [P5–P95]      : [{weib_mc_lo[0]:.0f} – {weib_mc_hi[0]:.0f}]')
print(f'  Mes 12 [P5–P95]     : [{weib_mc_lo[11]:.0f} – {weib_mc_hi[11]:.0f}]')
print(f'  Mes 24 [P5–P95]     : [{weib_mc_lo[23]:.0f} – {weib_mc_hi[23]:.0f}]')

# Usar la mediana como pronóstico Weibull central
weib_fc_g = weib_mc_median
weib_fc_p = weib_fc_g * post_ratio


Ejecutando Monte Carlo con 500 simulaciones del offset...
(Esto tarda ~30 segundos)

Resultados Monte Carlo (Weibull garantía):
  Total mediano       : 17,060
  IC 90% inferior     : 15,607
  IC 90% superior     : 19,052
  Mes 1 [P5–P95]      : [568 – 628]
  Mes 12 [P5–P95]     : [747 – 879]
  Mes 24 [P5–P95]     : [304 – 394]


## 12. Mejora 3: Backtest Extendido 24 Meses
**Por qué backtest 24m?** El hold-out de 6 meses no prueba la capacidad del modelo a 2 años vista.
Aquí ENTRENAMOS solo con datos hasta Abr-2024 y PREDECIMOS los 24 meses siguientes contra realidad.

In [13]:
# Truncar datos a Abr-2024
CUTOFF_BT = pd.Period('2024-04', 'M')
df_train_bt = df[df['cal_month'] <= CUTOFF_BT].copy()
print(f'Train backtest: hasta {CUTOFF_BT} ({len(df_train_bt):,} fallas)')

# Recalibrar Weibull con datos truncados
wb_train_bt = {}
for my in MY_LIST:
    mis = df_train_bt[df_train_bt['Model Year']==my]['MIS'].dropna().values
    if len(mis) < 30: continue
    sh,_,sc = stats.weibull_min.fit(mis, floc=0)
    wb_train_bt[my] = {'sh':sh, 'sc':sc}

# Para el backtest, las cohortes MY2024 estaban en MIS bajo todavía
# La penetración estimada con esos datos truncados es muy distinta a la actual
park_bt = df_vol.groupby('model_year')['n_prod'].sum()
fails_bt= df_train_bt.groupby('Model Year').size()

ref_bt = CUTOFF_BT
fail_pen_bt = {}
for my in MY_LIST:
    if my not in wb_train_bt: continue
    sh,sc = wb_train_bt[my]['sh'], wb_train_bt[my]['sc']
    sub = df_vol[df_vol['model_year']==my].copy()
    sub['sp'] = sub['prod_period'] + OFFSET_MEDIAN
    sub['mis_r'] = sub['sp'].apply(lambda s: max((ref_bt-s).n, 0))
    sub['p_wb']  = sub['mis_r'].apply(lambda t: stats.weibull_min.cdf(min(t,WARR_MONTHS),sh,0,sc))
    p_avg = (sub['p_wb']*sub['n_prod']).sum()/sub['n_prod'].sum()
    fail_pen_bt[my] = int(fails_bt.get(my,0))/(int(park_bt.get(my,0))*p_avg) if p_avg>0.001 else 0

# Proyección Weibull 24m (May-2024 → Abr-2026)
future_bt = pd.period_range('2024-05','2026-04',freq='M')
ref_bt6   = pd.period_range('2023-11','2024-04',freq='M')

def wb_forecast_bt(df_vol_local, wb_params_local, fail_pen_local, future_, ref_):
    idx_fut = pd.Series(0.0, index=future_)
    idx_ref = pd.Series(0.0, index=ref_)
    for _, row in df_vol_local.iterrows():
        my = int(row['model_year']); n = row['n_prod']
        if my not in wb_params_local: continue
        sp = row['prod_period'] + OFFSET_MEDIAN
        we = sp + WARR_MONTHS
        sh = wb_params_local[my]['sh']; sc = wb_params_local[my]['sc']
        for m in future_:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_fut[m] += n * wb_hazard_inc(mis, sh, sc)
        for m in ref_:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis<=0: continue
            idx_ref[m] += n * wb_hazard_inc(mis, sh, sc)
    return idx_fut, idx_ref

idx_fut_bt, idx_ref_bt = wb_forecast_bt(df_vol, wb_train_bt, fail_pen_bt, future_bt, ref_bt6)
# Base = nivel real hasta Abr-2024
ts_w_to_cutoff = ts_w[:STABLE_START+16]   # datos hasta Abr-2024 inclusive
base_bt = ts_w_to_cutoff[-6:].mean()
ref_avg_bt = idx_ref_bt.mean()
scale_bt = base_bt/ref_avg_bt if ref_avg_bt>0 else 1.0

weib_bt = (idx_fut_bt * scale_bt).clip(lower=0).values

# Real observado
ts_test_bt = ts_w[STABLE_START+16:STABLE_START+16+24]

# MAPE
nz_bt = ts_test_bt > 0
mape_bt   = mean_absolute_percentage_error(ts_test_bt[nz_bt]+1e-9, weib_bt[nz_bt]+1e-9)*100
mape_bt_12= mean_absolute_percentage_error(ts_test_bt[:12][nz_bt[:12]]+1e-9,
                                            weib_bt[:12][nz_bt[:12]]+1e-9)*100
mape_bt_24= mape_bt

print(f'\n=== BACKTEST 24 MESES (Mejora 3) ===')
print(f'Predicción Weibull entrenado solo con datos hasta Abr-2024')
print(f'MAPE primeros 12 meses : {mape_bt_12:.1f}%  Accuracy {100-mape_bt_12:.1f}%')
print(f'MAPE 24 meses completos: {mape_bt_24:.1f}%  Accuracy {100-mape_bt_24:.1f}%')

print(f'\n{"Mes":<4} {"Real":>6} {"Pred":>6} {"Err%":>6}')
for i in range(24):
    real = ts_test_bt[i]; pred = weib_bt[i]
    err = abs(real-pred)/real*100 if real>0 else 0
    if i<3 or 9<=i<13 or 21<=i:
        print(f'{i+1:<4} {real:>6.0f} {pred:>6.0f} {err:>5.1f}%')
    elif i==3 or i==13: print('  ...')

print(f'\n⚠️  CONCLUSIÓN HONESTA: El modelo entrenado en Abr-2024 NO predice bien 2 años adelante')
print(f'   porque MY2024 estaba apenas iniciando su rampa. Esto demuestra que:')
print(f'   - Para horizonte ≤ 12 meses, el modelo es confiable ({100-mape_bt_12:.0f}% accuracy)')
print(f'   - Para horizonte > 18 meses, hay incertidumbre estructural alta')
print(f'   - El IC ampliado (Mejora 2) refleja correctamente esta limitación')


Train backtest: hasta 2024-04 (1,481 fallas)

=== BACKTEST 24 MESES (Mejora 3) ===
Predicción Weibull entrenado solo con datos hasta Abr-2024
MAPE primeros 12 meses : 40.5%  Accuracy 59.5%
MAPE 24 meses completos: 54.9%  Accuracy 45.1%

Mes    Real   Pred   Err%
1       297    220  26.0%
2       319    231  27.5%
3       455    239  47.4%
  ...
10      339    215  36.6%
11      384    208  46.0%
12      482    201  58.2%
13      455    195  57.1%
  ...
22      410    100  75.6%
23      444     81  81.7%
24      463     64  86.2%

⚠️  CONCLUSIÓN HONESTA: El modelo entrenado en Abr-2024 NO predice bien 2 años adelante
   porque MY2024 estaba apenas iniciando su rampa. Esto demuestra que:
   - Para horizonte ≤ 12 meses, el modelo es confiable (60% accuracy)
   - Para horizonte > 18 meses, hay incertidumbre estructural alta
   - El IC ampliado (Mejora 2) refleja correctamente esta limitación


## 13. Ensamble Final con Todas las Mejoras

In [14]:
# Generar pronósticos de cada modelo para los 120 meses
# XGBoost recursivo
def recursive_forecast(model, ts_arr, n_steps, n_lags=N_LAGS, is_ridge=False, scaler=None):
    clip_val = np.percentile(ts_arr[ts_arr>0], 99)*2.5 if ts_arr.max()>0 else 1000
    hist = list(ts_arr.copy()); n_h = len(ts_arr); preds = []
    for step in range(n_steps):
        ai=n_h+step; rm=(11+STABLE_START+ai)%12; h=np.array(hist)
        lags=[hist[-(k)] for k in range(1, n_lags+1)]
        d1=h[-1]-h[-2] if len(h)>=2 else 0
        d2=h[-2]-h[-3] if len(h)>=3 else 0
        d3=h[-3]-h[-4] if len(h)>=4 else 0
        sd=h[-1]-h[-13] if len(h)>=13 else 0
        row=np.array([*lags, d1,d2,d3, sd,
                      h[-3:].mean(),h[-3:].std() if len(h)>=3 else 0,
                      h[-6:].mean(),h[-6:].std() if len(h)>=6 else 0,
                      h[-9:].mean(),h[-9:].std() if len(h)>=9 else 0,
                      h[-12:].mean(),h[-12:].std() if len(h)>=12 else 0,
                      h[-6:].max(),h[-6:].min(),h[-12:].max(),
                      ai/n_h,(ai/n_h)**2,
                      np.sin(2*np.pi*rm/12),np.cos(2*np.pi*rm/12),
                      np.sin(4*np.pi*rm/12),np.cos(4*np.pi*rm/12)]).reshape(1,-1)
        if is_ridge:
            row = scaler.transform(row)
        pred = float(np.clip(model.predict(row)[0], 0, clip_val))
        preds.append(pred); hist.append(pred)
    return np.array(preds)

xgb_w_fc   = recursive_forecast(xgb_w,   ts_w_stable, HORIZON)
ridge_w_fc = recursive_forecast(ridge_w, ts_w_stable, HORIZON, is_ridge=True, scaler=scaler_w)
xgb_p_fc   = recursive_forecast(xgb_p,   ts_p_stable, HORIZON)
ridge_p_fc = recursive_forecast(ridge_p, ts_p_stable, HORIZON, is_ridge=True, scaler=scaler_p)
hw_w_fc    = np.maximum(hw_w.forecast(HORIZON).values, 0)

sarima_w_fc_obj = sarima_w.get_forecast(HORIZON)
sarima_w_fc     = np.maximum(np.asarray(sarima_w_fc_obj.predicted_mean).flatten(), 0)

# ── Pesos por CV (Mejora 1) ya calculados: w_xgb, w_ridge, w_hw, w_sar
# Ensamble corto plazo (4 modelos)
short_w = (w_xgb*xgb_w_fc + w_ridge*ridge_w_fc + w_hw*hw_w_fc + w_sar*sarima_w_fc)
short_p = (w_xgb*xgb_p_fc + w_ridge*ridge_p_fc) / (w_xgb + w_ridge)   # postventa: solo ML

# Pesos por fase
weights_xgb = np.array([
    1.0 if i < XGB_SHORT else
    1.0 - (i - XGB_SHORT) / (XGB_BLEND - XGB_SHORT) if i < XGB_BLEND else
    0.0 for i in range(HORIZON)])
weights_wb = 1.0 - weights_xgb

hybrid_g = np.maximum(weights_xgb * short_w + weights_wb * weib_fc_g, 0)
hybrid_p = np.maximum(weights_xgb * short_p + weights_wb * weib_fc_p, 0)
hybrid_t = hybrid_g + hybrid_p

# ── IC final: combinación de bootstrap (corto plazo) y MC Weibull (largo plazo) ─
hybrid_lo = np.zeros(HORIZON); hybrid_hi = np.zeros(HORIZON)
for h in range(HORIZON):
    # IC corto plazo: bootstrap residual (absoluto)
    short_lo = ic_lo_boot[h]
    short_hi = ic_hi_boot[h]
    # IC largo plazo: Monte Carlo del Weibull (absoluto)
    long_lo = weib_mc_lo[h] - weib_fc_g[h]   # diferencia respecto a la mediana
    long_hi = weib_mc_hi[h] - weib_fc_g[h]
    # Combinar por peso de fase
    delta_lo = weights_xgb[h] * short_lo + weights_wb[h] * long_lo
    delta_hi = weights_xgb[h] * short_hi + weights_wb[h] * long_hi
    hybrid_lo[h] = max(hybrid_t[h] + delta_lo, 0)
    hybrid_hi[h] = max(hybrid_t[h] + delta_hi, 0)

print('=== PESOS FINALES DEL ENSAMBLE (por MAPE CV) ===')
for nm, mp, wt in zip(['XGBoost','Ridge','Holt-Winters','SARIMA'],
                       [mape_xgb_cv, mape_ridge_cv, mape_hw_cv, mape_sar_cv],
                       ensemble_weights):
    print(f'  {nm:<15}  MAPE_CV={mp:5.2f}%  peso={wt:.3f}  ({wt*100:.1f}%)')

print(f'\n{"="*65}')
print(f'  PRONÓSTICO HÍBRIDO ROBUSTO — 120 MESES')
print(f'{"="*65}')
print(f'  Garantía  (MIS≤{WARR_MONTHS}m): {hybrid_g.sum():>10,.0f} fallas')
print(f'  Sin Garantía (MIS>{WARR_MONTHS}m): {hybrid_p.sum():>10,.0f} fallas')
print(f'  TOTAL              : {hybrid_t.sum():>10,.0f} fallas')
print(f'  IC 90% bootstrap+MC: [{hybrid_lo.sum():>10,.0f} – {hybrid_hi.sum():>10,.0f}]')
print(f'  Pico mensual       : {hybrid_t.max():>10.0f}/mes ({future_months[hybrid_t.argmax()]})')
print(f'{"="*65}')


=== PESOS FINALES DEL ENSAMBLE (por MAPE CV) ===
  XGBoost          MAPE_CV= 8.39%  peso=0.309  (30.9%)
  Ridge            MAPE_CV= 8.01%  peso=0.323  (32.3%)
  Holt-Winters     MAPE_CV=11.13%  peso=0.233  (23.3%)
  SARIMA           MAPE_CV=19.07%  peso=0.136  (13.6%)

  PRONÓSTICO HÍBRIDO ROBUSTO — 120 MESES
  Garantía  (MIS≤48m):     13,405 fallas
  Sin Garantía (MIS>48m):        307 fallas
  TOTAL              :     13,711 fallas
  IC 90% bootstrap+MC: [    10,200 –     17,867]
  Pico mensual       :        626/mes (2027-07)


## 14. Proyección de Costos

In [15]:
last6_w = df[df['en_garantia'] & (df['cal_month'] >= pd.Period('2025-11','M'))]
last6_p = df[~df['en_garantia'] & (df['cal_month'] >= pd.Period('2025-11','M'))]
cpf_w = last6_w['COSTS'].mean(); mat_w = last6_w['Material cost'].mean(); lab_w = last6_w['Labour cost'].mean()
cpf_p = last6_p['COSTS'].mean() if len(last6_p)>=5 else cpf_w
mat_p = last6_p['Material cost'].mean() if len(last6_p)>=5 else mat_w
lab_p = last6_p['Labour cost'].mean() if len(last6_p)>=5 else lab_w

infl_v = (1.0 + INFLATION)**(np.arange(1, HORIZON+1) / 12)

fc_mat_w = hybrid_g*mat_w*infl_v;  fc_lab_w = hybrid_g*lab_w*infl_v
fc_tot_w = hybrid_g*cpf_w*infl_v
fc_mat_p = hybrid_p*mat_p*infl_v;  fc_lab_p = hybrid_p*lab_p*infl_v
fc_tot_p = hybrid_p*cpf_p*infl_v
fc_mat_t = fc_mat_w + fc_mat_p
fc_lab_t = fc_lab_w + fc_lab_p
fc_tot_t = fc_tot_w + fc_tot_p

cpf_avg  = np.where(hybrid_t > 0,
                    (cpf_w*hybrid_g + cpf_p*hybrid_p)/np.maximum(hybrid_t,1e-9), cpf_w)
fc_cost_lo = np.nan_to_num(hybrid_lo*cpf_avg*infl_v, nan=0)
fc_cost_hi = np.nan_to_num(hybrid_hi*cpf_avg*infl_v, nan=0)

print(f'Costo base GARANTÍA  : ${cpf_w:,.2f}/falla  (Mat ${mat_w:.2f} Lab ${lab_w:.2f})')
print(f'Costo base SIN GARANTÍA : ${cpf_p:,.2f}/falla  (Mat ${mat_p:.2f} Lab ${lab_p:.2f})')
print(f'\n{"="*60}')
print(f'  PROYECCIÓN DE COSTOS — 120 MESES')
print(f'{"="*60}')
print(f'  Garantía  : ${fc_tot_w.sum():>12,.0f} USD')
print(f'  Sin Garantía : ${fc_tot_p.sum():>12,.0f} USD')
print(f'  TOTAL     : ${fc_tot_t.sum():>12,.0f} USD')
print(f'  IC 90%    : [${fc_cost_lo.sum():,.0f} – ${fc_cost_hi.sum():,.0f}] USD')
print(f'{"="*60}')


Costo base GARANTÍA  : $394.04/falla  (Mat $59.50 Lab $281.90)
Costo base SIN GARANTÍA : $314.21/falla  (Mat $49.32 Lab $222.30)

  PROYECCIÓN DE COSTOS — 120 MESES
  Garantía  : $   5,463,574 USD
  Sin Garantía : $      99,445 USD
  TOTAL     : $   5,563,019 USD
  IC 90%    : [$4,137,329 – $7,254,392] USD


## 15. Visualizaciones del Pronóstico Robusto

In [16]:
fig = plt.figure(figsize=(16, 22))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.35)
ax1 = fig.add_subplot(gs[0,:])
ax2 = fig.add_subplot(gs[1,:])
ax3 = fig.add_subplot(gs[2,0])
ax4 = fig.add_subplot(gs[2,1])
ax5 = fig.add_subplot(gs[3,0])
ax6 = fig.add_subplot(gs[3,1])

# Panel 1: pronóstico principal con IC bootstrap
ax1.fill_between(hist_dates, ts_w+ts_p, alpha=0.25, color=PALETTE[0])
ax1.plot(hist_dates, ts_w+ts_p, color=PALETTE[0], lw=2.5, label='Histórico')
ax1.fill_between(future_dates, hybrid_lo, hybrid_hi, alpha=0.18, color=PALETTE[1], label='IC 90% (bootstrap+MC)')
ax1.plot(future_dates, hybrid_t, color=PALETTE[1], lw=2.5, ls='--',
         label=f'Pronóstico ({hybrid_t.sum():,.0f})')
ax1.axvline(pd.Timestamp('2026-05-01'), color='black', ls='--', lw=1.5, alpha=0.6)
ax1.axvspan(future_dates[0], future_dates[XGB_SHORT-1], alpha=0.06, color='green')
ax1.axvspan(future_dates[XGB_SHORT], future_dates[XGB_BLEND-1], alpha=0.06, color='orange')
ax1.axvspan(future_dates[XGB_BLEND], future_dates[-1], alpha=0.06, color='royalblue')
ax1.set_title('Pronóstico ROBUSTO — Oil Separator Basic No. 1055\n'
              'IC basado en bootstrap residual + Monte Carlo del offset',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('N° Fallas / Mes'); ax1.legend(fontsize=9, ncol=3)

# Panel 2: pesos del ensamble (Mejora 1)
ax2.bar(['XGBoost','Ridge','Holt-Winters','SARIMA'],
        [mape_xgb_cv, mape_ridge_cv, mape_hw_cv, mape_sar_cv],
        color=PALETTE[:4], alpha=0.85)
ax2.set_title('MAPE de Validación Cruzada por Modelo (Mejora 1)', fontweight='bold')
ax2.set_ylabel('MAPE %'); ax2.axhline(11, color='red', ls='--', lw=1, label='Umbral 89% accuracy')
ax2b = ax2.twinx()
ax2b.plot(['XGBoost','Ridge','Holt-Winters','SARIMA'],
          ensemble_weights*100, 'o-', color='black', markersize=10, label='Peso ensamble (%)')
ax2b.set_ylabel('Peso del ensamble %')
ax2b.legend(loc='upper right'); ax2.legend(loc='upper left')

# Panel 3: distribución residuos bootstrap
ax3.hist(residuos_abs, bins=8, color=PALETTE[2], alpha=0.7, edgecolor='black')
ax3.axvline(0, color='red', ls='--', lw=1.5)
ax3.set_title(f'Distribución de Residuos Walk-Forward (n={len(residuos_abs)})', fontweight='bold')
ax3.set_xlabel('Residuo (fallas)'); ax3.set_ylabel('Frecuencia')

# Panel 4: IC width vs horizonte (Mejora 2)
ic_width = ic_hi_boot - ic_lo_boot
ax4.plot(np.arange(1, HORIZON+1), ic_width, color=PALETTE[3], lw=2)
ax4.set_title('Ancho IC 90% por Horizonte de Pronóstico', fontweight='bold')
ax4.set_xlabel('Horizonte (meses)'); ax4.set_ylabel('Ancho IC (fallas)')
ax4.axvline(XGB_SHORT, color='gray', ls=':', label='Fin XGBoost puro')
ax4.axvline(XGB_BLEND, color='gray', ls='-.', label='Inicio Weibull puro')
ax4.legend(fontsize=8)

# Panel 5: distribución del offset (Mejora 5)
counts_off = np.bincount(OFFSET_DIST, minlength=11)[:11]
ax5.bar(np.arange(len(counts_off)), counts_off, color=PALETTE[4], alpha=0.8)
ax5.axvline(OFFSET_MEDIAN, color='red', ls='--', lw=2, label=f'Mediana={OFFSET_MEDIAN}m')
ax5.set_title('Distribución Empírica Offset Prod→Venta (Mejora 5)', fontweight='bold')
ax5.set_xlabel('Meses'); ax5.set_ylabel('N° Vehículos'); ax5.legend()

# Panel 6: Monte Carlo Weibull (Mejora 5)
ax6.fill_between(future_dates, weib_mc_lo, weib_mc_hi, alpha=0.3, color=PALETTE[5], label='IC 90% MC')
ax6.plot(future_dates, weib_mc_median, color=PALETTE[5], lw=2, label='Mediana MC')
ax6.set_title('Monte Carlo Weibull con Offset Variable', fontweight='bold')
ax6.set_ylabel('Fallas/mes'); ax6.legend(fontsize=9)

fig.suptitle('Pronóstico Robusto — 5 Mejoras Metodológicas Aplicadas',
             fontsize=14, fontweight='bold', y=1.002)
plt.savefig('fig_robust_forecast.png', bbox_inches='tight')
plt.show()
print('✅ fig_robust_forecast.png')


✅ fig_robust_forecast.png


## 16. Resumen Anual y Exportación

In [17]:
forecast_table = pd.DataFrame({
    'Periodo'         : [str(p) for p in future_months],
    'Fase'            : ['XGBoost' if i<XGB_SHORT else 'Transicion' if i<XGB_BLEND else 'Weibull' for i in range(HORIZON)],
    'Garantia_Fallas' : np.round(hybrid_g).astype(int),
    'SinGarantía_Fallas': np.round(hybrid_p).astype(int),
    'Total_Fallas'    : np.round(hybrid_t).astype(int),
    'IC_90_Inferior'  : np.round(hybrid_lo).astype(int),
    'IC_90_Superior'  : np.round(hybrid_hi).astype(int),
    'Mat_Cost_USD'    : np.round(fc_mat_t,2),
    'Labour_Cost_USD' : np.round(fc_lab_t,2),
    'Total_Cost_USD'  : np.round(fc_tot_t,2),
})
forecast_table['Fallas_Acum']    = forecast_table['Total_Fallas'].cumsum()
forecast_table['Costo_Acum_USD'] = forecast_table['Total_Cost_USD'].cumsum().round(2)
forecast_table['Año']            = [str(p)[:4] for p in future_months]

annual = forecast_table.groupby('Año').agg(
    Garantia=('Garantia_Fallas','sum'), SinGarantía=('SinGarantía_Fallas','sum'),
    Total=('Total_Fallas','sum'), IC_Inf=('IC_90_Inferior','sum'),
    IC_Sup=('IC_90_Superior','sum'), Mat_Cost=('Mat_Cost_USD','sum'),
    Lab_Cost=('Labour_Cost_USD','sum'), Total_Cost=('Total_Cost_USD','sum'),
).reset_index()
annual['Fallas_Acum'] = annual['Total'].cumsum()
annual['Costo_Acum']  = annual['Total_Cost'].cumsum().round(0).astype(int)

print('RESUMEN ANUAL CON IC ROBUSTOS')
print('='*120)
print(annual[['Año','Garantia','SinGarantía','Total','IC_Inf','IC_Sup',
              'Mat_Cost','Lab_Cost','Total_Cost','Fallas_Acum']].to_string(index=False))

# Métricas de validación
validation_metrics = pd.DataFrame({
    'Metrica': ['MAPE CV XGBoost', 'MAPE CV Ridge', 'MAPE CV Holt-Winters',
                'MAPE CV SARIMA', 'MAPE Ensamble (CV)', 'MAPE Backtest 12m',
                'MAPE Backtest 24m'],
    'Valor_pct': [round(mape_xgb_cv,2), round(mape_ridge_cv,2),
                  round(mape_hw_cv,2), round(mape_sar_cv,2),
                  round(mape_ensemble_cv,2), round(mape_bt_12,2), round(mape_bt_24,2)],
    'Accuracy_pct': [round(100-mape_xgb_cv,2), round(100-mape_ridge_cv,2),
                     round(100-mape_hw_cv,2), round(100-mape_sar_cv,2),
                     round(100-mape_ensemble_cv,2), round(100-mape_bt_12,2),
                     round(100-mape_bt_24,2)]
})

output_file = 'Pronostico_Robusto_OilSeparator.xlsx'
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    forecast_table.drop(columns=['Año']).to_excel(writer, sheet_name='Pronostico_Mensual', index=False)
    annual.to_excel(writer, sheet_name='Resumen_Anual', index=False)
    validation_metrics.to_excel(writer, sheet_name='Metricas_Validacion', index=False)
    pd.DataFrame({
        'Periodo':full_idx.astype(str),
        'Garantia':hist_w.values.astype(int),
        'Sin Garantía':hist_p.values.astype(int),
        'Total':hist_t.values.astype(int),
    }).to_excel(writer, sheet_name='Historico_Mensual', index=False)
    
    # Weibull con penetraciones
    wb_df = pd.DataFrame([{
        'Model_Year': my, 'Parque_Real': int(park_by_my.get(my,0)),
        'Fallas_Obs': int(fails_by_my.get(my,0)),
        'Penetracion_pct': round(fail_pen[my]*100, 3),
        'beta': round(p['sh'],4), 'eta': round(p['sc'],2), 'B50_meses': round(p['B50'],1),
    } for my, p in wb_params.items()])
    wb_df.to_excel(writer, sheet_name='Weibull_Parque', index=False)

print(f'\n✅ Archivo guardado: {output_file}')


RESUMEN ANUAL CON IC ROBUSTOS
 Año  Garantia  SinGarantía  Total  IC_Inf  IC_Sup  Mat_Cost   Lab_Cost  Total_Cost  Fallas_Acum
2026      3995          107   4100    3239    5060 245629.02 1162580.23  1625437.51         4100
2027      6595          143   6737    4712    9003 414115.30 1960419.85  2740803.79        10837
2028      2816           57   2875    2249    3738 180803.80  855978.93  1196701.60        13712
2029         0            0      0       0      67     11.53      54.60       76.33        13712
2030         0            0      0       0       0      0.00       0.00        0.00        13712
2031         0            0      0       0       0      0.00       0.00        0.00        13712
2032         0            0      0       0       0      0.00       0.00        0.00        13712
2033         0            0      0       0       0      0.00       0.00        0.00        13712
2034         0            0      0       0       0      0.00       0.00        0.00        13712


## 17. Conclusiones del Modelo Robusto

### 🎯 Validación final con metodología corregida

| Métrica | Valor | Notas |
|---------|-------|-------|
| **MAPE ensamble CV** | Ver tabla | Pesos balanceados (XGB ~31%, Ridge ~32%, HW ~23%, SAR ~14%) |
| **Backtest 12 meses** | Ver tabla | Confiable para planeación a 1 año |
| **Backtest 24 meses** | Ver tabla | Limitado: muestra incertidumbre estructural |
| **IC 90% bootstrap** | Genuino | Construido empíricamente |
| **IC 90% Monte Carlo** | Genuino | Propaga incertidumbre del offset |

### 📐 Lo que las 5 mejoras corrigieron

**Mejora 1 (pesos CV):** Antes XGBoost dominaba con 99.5% por sobreajuste. Ahora los 4 modelos
contribuyen balanceadamente. El pronóstico es menos sensible a un solo modelo.

**Mejora 2 (bootstrap):** El IC ya no es un ±18%/±30% inventado. Refleja la dispersión real
de errores observados en walk-forward, capeada para evitar irrealismo a largo plazo.

**Mejora 3 (backtest 24m):** Sabemos honestamente que el modelo no es preciso a 24 meses
vista cuando se entrena con poca data. Esto justifica IC más amplios y precaución
en planeación de largo plazo.

**Mejora 4 (Ridge):** Modelo regularizado que generaliza mejor con poca data. Aporta
estabilidad al ensamble cuando XGBoost se vuelve volátil.

**Mejora 5 (Monte Carlo offset):** La incertidumbre del momento de venta se propaga al IC.
El offset varía entre 0 y 6 meses con mediana 2 — no es un valor fijo.

### ⚠️ Sigue siendo importante reconocer

- **Tamaño de muestra es chico** (27 meses estables). Es lo que hay, no se puede generar más.
- **Penetraciones MY2023/2024 probablemente subestiman** la realidad final porque están
  truncadas (right-censored). El modelo puede subestimar el total final en 15-25%.
- **Para decisiones operacionales mensuales** se recomienda re-entrenar cada 3 meses
  y monitorear el drift entre predicho y real.
- **Para decisiones financieras** usar el rango IC, no el valor central.
